TODO:
1) Handle missing values (fill NaN values)
2) Normalize the data (scaling) 
3) Select the most important features
4) Train the preprocessing pipeline


Chosen features:
1) soil_nitrogen
2) soil_pH 
3) soil_potassium
4) soil_moisture 
5) crop_height



In [65]:
# Setup
import pandas as pd
import numpy as np
import random
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor


In [8]:
# Making a dataset generator
def noisy_linear_function(x, weight, bias, noise):
    return x * weight + bias + random.uniform(-noise,noise)

In [41]:
# Making placeholder for data 

soil_nitrogen = [] # ppm
soil_pH = [] # standard oH scale
soil_potassium = [] # ppm 
soil_moisture = [] # percentage (%)
crop_height = [] # cm

In [42]:
# Dataset generation
for i in range(0, 400):
    soil_nitrogen.append(noisy_linear_function(i,0.2, 15, 0.3))
    soil_pH.append(noisy_linear_function(i, 0.01, 5, 0.2))
    soil_potassium.append(noisy_linear_function(i, 0.25, 50, 0.3))
    soil_moisture.append(noisy_linear_function(i, 0.3, 15, 0.2))
    crop_height.append(noisy_linear_function(i, 0.4, 20, 0.1))

In [43]:
dataset = {
    "soil_nitrogen": soil_nitrogen,
    "soil_pH": soil_pH,
    "soil_potassium": soil_potassium,
    "soil_moisture": soil_moisture,
    "crop_height": crop_height
}

In [44]:
df = pd.DataFrame(dataset)

In [45]:
# Checking data
df.head()

,soil_nitrogen,soil_pH,soil_potassium,soil_moisture,crop_height
0,15.293794,5.140310,50.106537,15.036246,20.062518
1,15.198153,5.081757,49.984561,15.461687,20.493108
2,15.263873,4.960142,50.430187,15.793382,20.717773
3,15.814514,5.116269,50.877233,16.077570,21.250307
4,15.943003,5.004064,50.845388,16.278194,21.588409


In [48]:
# Introducing missing values (NaN)
mask = np.random.rand(*df.shape) < 0.1
df[mask] = np.nan


In [50]:
# Check missing values 
df.isna().sum()

soil_nitrogen     32
soil_pH           42
soil_potassium    49
soil_moisture     44
crop_height       43
dtype: int64

# 2. Data preprocessing and pipeline making

In [51]:
# Separate features and target 
X = df.drop("crop_height", axis=1)
y = df["crop_height"]

In [52]:
# Fill missing values (imputation)

imputer = SimpleImputer(strategy="mean")
X_filled = imputer.fit_transform(X)
X_filled_df = pd.DataFrame(X_filled, columns=X.columns)
X_filled_df

,soil_nitrogen,soil_pH,soil_potassium,soil_moisture
0,15.293794,5.140310,50.106537,15.036246
1,15.198153,5.081757,49.984561,74.831685
2,54.421434,7.038704,50.430187,15.793382
3,15.814514,5.116269,50.877233,74.831685
4,15.943003,5.004064,50.845388,74.831685
...,...,...,...,...
395,94.205770,8.797296,148.615432,74.831685
396,94.230600,8.999013,149.164174,133.604358
397,94.206900,7.038704,149.418757,134.243228
398,54.421434,9.016663,149.685700,74.831685


In [58]:
# filling in missing values in y
df["crop_height"] = df["crop_height"].fillna(df["crop_height"].mean())

In [62]:
y = df["crop_height"]

In [60]:
df.isna().sum()

soil_nitrogen     32
soil_pH           42
soil_potassium    49
soil_moisture     44
crop_height        0
dtype: int64

In [53]:
# Normalize (scaling)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled_df)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled_df

,soil_nitrogen,soil_pH,soil_potassium,soil_moisture
0,-1.786599,-1.743223,-1.805711,-1.831037
1,-1.790966,-1.796990,-1.810157,0.000000
2,0.000000,0.000000,-1.793913,-1.807852
3,-1.762823,-1.765298,-1.777617,0.000000
4,-1.756956,-1.868331,-1.778778,0.000000
...,...,...,...,...
395,1.816585,1.614847,1.785150,0.000000
396,1.817718,1.800076,1.805153,1.799718
397,1.816636,0.000000,1.814433,1.819281
398,0.000000,1.816284,1.824164,0.000000


In [63]:
selector = SelectKBest(score_func=f_regression, k=2)

X_selected = selector.fit_transform(X_scaled_df, y)

selected_features = X.columns[selector.get_support()]
print('Most important features:', list(selected_features))

Most important features: ['soil_potassium', 'soil_moisture']


In [ ]:
# Creating pipeline 
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor())
])

In [ ]:
# Training pipeline
pipeline.fit(X,y)

,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'mean'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True
